In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from rosbags.highlevel import AnyReader
from sklearn.linear_model import RANSACRegressor, LinearRegression

# =========================
# SETTINGS
# =========================

#BAG_PATH = Path(r"C:\ROSBAGS VERWIJDER NA BEP\rosbag_0.mcap")

BAG_PATH = Path(r"c:\rosbag_0.mcap")
POINT_TOPIC = "/rslidar/M1P_deskewed"

MAX_FRAMES_TO_PROCESS = 3
RANDOM_DOWNSAMPLE_POINTS = 10000

# =========================
# POINTCLOUD2 PARSER
# =========================

def parse_pc2(msg):
    """
    Converts ROS PointCloud2 message to Nx3 numpy array: x, y, z.
    """
    field_offsets = {field.name: field.offset for field in msg.fields}

    required_fields = ["x", "y", "z"]
    for field in required_fields:
        if field not in field_offsets:
            raise ValueError(f"PointCloud2 is missing field: {field}")

    point_step = msg.point_step
    data = msg.data

    dtype = np.dtype({
        "names": ["x", "y", "z"],
        "formats": ["<f4", "<f4", "<f4"],
        "offsets": [
            field_offsets["x"],
            field_offsets["y"],
            field_offsets["z"],
        ],
        "itemsize": point_step,
    })

    points = np.frombuffer(data, dtype=dtype)
    xyz = np.vstack((points["x"], points["y"], points["z"])).T

    # Remove NaN / infinite points
    xyz = xyz[np.isfinite(xyz).all(axis=1)]
    return xyz

# =========================
# DOWNSAMPLING
# =========================

def random_downsample(points, max_points=10000):
    if len(points) <= max_points:
        return points
    idx = np.random.choice(len(points), max_points, replace=False)
    return points[idx]

# =========================
# PLOTTING
# =========================

def plot_birds_eye(points, ground_points, frame_id):
    plt.figure(figsize=(8, 8))
    
    # Plot all points in gray
    plt.scatter(
        points[:, 0],
        points[:, 1],
        s=1,
        c="lightgray",
        label="All points"
    )

    # Overlay ground points in green
    if len(ground_points) > 0:
        plt.scatter(
            ground_points[:, 0],
            ground_points[:, 1],
            s=2,
            c="green",
            label="Ground points"
        )

    plt.title(f"Bird's-Eye View - Frame {frame_id}")
    plt.xlabel("X (Forward)")
    plt.ylabel("Y (Left/Right)")
    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.show()

# =========================
# GROUND SEGMENTATION
# =========================

class GroundSegmentation:
    def __init__(
        self,
        distance_threshold=0.05,
        section_length=2.0,
        min_points=30,
        epsilon=0.20,
        gamma_deg=10,
    ):
        self.distance_threshold = distance_threshold
        self.section_length = section_length
        self.min_points = min_points
        self.epsilon = epsilon
        self.gamma = np.radians(gamma_deg)

    def split_quadrants(self, points):
        x, y = points[:, 0], points[:, 1]
        return [
            points[x >= 0], # Front
            points[x < 0],  # Rear
            points[y >= 0], # Left
            points[y < 0]   # Right
        ]

    def split_sections(self, points):
        if len(points) == 0:
            return []
        r = np.sqrt(points[:, 0] ** 2 + points[:, 1] ** 2)
        r_max = np.max(r)
        sections = []
        start = 0.0
        while start < r_max:
            mask = (r >= start) & (r < start + self.section_length)
            section = points[mask]
            if len(section) > 0:
                sections.append(section)
            start += self.section_length
        return sections

    def fit_plane_ransac(self, section):
        model = RANSACRegressor(
            estimator=LinearRegression(),
            residual_threshold=self.distance_threshold,
            min_samples=3,
            random_state=42,
        )
        model.fit(section[:, :2], section[:, 2])
        return model

    def plane_normal(self, model):
        a, b = model.estimator_.coef_
        normal = np.array([-a, -b, 1.0])
        normal /= np.linalg.norm(normal)
        return normal

    def check_continuity(self, current_model, prev_model):
        delta_h = abs(current_model.estimator_.intercept_ - prev_model.estimator_.intercept_)
        n_curr = self.plane_normal(current_model)
        n_prev = self.plane_normal(prev_model)
        angle = np.arccos(np.clip(np.dot(n_curr, n_prev), -1.0, 1.0))
        return delta_h < self.epsilon and angle < self.gamma

    def get_inliers(self, section, model):
        z_pred = model.predict(section[:, :2])
        error = np.abs(section[:, 2] - z_pred)
        return section[error < self.distance_threshold]

    def algorithm_2_ransac(self, points):
        quadrants = self.split_quadrants(points)
        final_ground = []

        for quad_points in quadrants:
            sections = self.split_sections(quad_points)
            prev_model = None

            for section in sections:
                if len(section) < self.min_points:
                    continue
                try:
                    model = self.fit_plane_ransac(section)
                    if prev_model is not None:
                        if not self.check_continuity(model, prev_model):
                            model = prev_model
                    
                    inliers = self.get_inliers(section, model)
                    if len(inliers) > 0:
                        final_ground.append(inliers)
                    prev_model = model
                except:
                    continue

        return np.vstack(final_ground) if final_ground else np.empty((0, 3))

# =========================
# MCAP PROCESSING
# =========================

def process_mcap(path):
    segmenter = GroundSegmentation()

    print(f"Opening bag: {path}")
    with AnyReader([path]) as reader:
        connections = [c for c in reader.connections if c.topic == POINT_TOPIC]

        if not connections:
            print("Topic not found. Available topics:")
            for c in reader.connections: print(f" - {c.topic}")
            return

        frame_count = 0

        for connection, timestamp, rawdata in reader.messages(connections=connections):
            # STOP condition for the first 10 frames
            if frame_count >= MAX_FRAMES_TO_PROCESS:
                print(f"\nFinished processing {MAX_FRAMES_TO_PROCESS} frames.")
                break

            msg = reader.deserialize(rawdata, connection.msgtype)
            full_points = parse_pc2(msg)
            
            # Optional: Downsample for speed/visualization
            points = random_downsample(full_points, RANDOM_DOWNSAMPLE_POINTS)

            ground_points = segmenter.algorithm_2_ransac(points)

            print(f"Frame {frame_count} | Ground: {len(ground_points)}/{len(points)} pts")
            
            # Show the plot
            plot_birds_eye(points, ground_points, frame_count)

            frame_count += 1

if __name__ == "__main__":
    process_mcap(BAG_PATH)